### Source Tables:
- _exponent._bronze_allscripts_tw_works.dbo_medication
- _exponent._bronze_allscripts_tw_works.dbo_medication_de
- _exponent._bronze_allscripts_tw_works.dbo_item_medication (for PatientID linkage)

### Strategy:
- Join dbo_medication with dbo_medication_de via MedDictDE to get RxNorm codes
- Map RxNormCode/RxNormCodeNormalized to OMOP drug_concept_id
- Fallback: Map NDC to RxNorm via concept_relationship
- Calculate drug_exposure_end_date from start + DaysSupply
- Map route codes via domain_source_to_concept
- Use drug_type_concept_id based on context (prescription=32838, admin=32818)

### Notes:
- This notebook depends on source_to_person being populated
- provider_id and visit_occurrence_id will be NULL (mapping tables not yet created)
- RxNorm concept mapping requires OMOP vocabulary tables to be loaded
- dbo_medication does not have PatientID directly - linked via dbo_item_medication

In [ ]:
source = 'allscripts_tw'

# Transformation

In [ ]:
silver_drug_exposure_df = spark.sql(f'''
SELECT
  -- Drug concept mapping: RxNorm code to standard OMOP concept
  -- Priority: RxNormCodeNormalized > RxNormCode > NDC mapping > 0
  COALESCE(
    rxnorm_concept.concept_id,
    rxnorm_from_ndc.concept_id,
    0
  ) AS drug_concept_id,
  
  -- Start date
  DATE(COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM)) AS drug_exposure_start_date,
  COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM) AS drug_exposure_start_datetime,
  
  -- End date: explicit end or calculated from start + days supply
  CASE
    WHEN med.EndFuzzySortAs IS NOT NULL THEN DATE(med.EndFuzzySortAs)
    WHEN med.DaysSupply IS NOT NULL AND med.DaysSupply > 0 
      THEN DATE_ADD(DATE(COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM)), med.DaysSupply)
    WHEN med.DaysToTake IS NOT NULL AND med.DaysToTake > 0 
      THEN DATE_ADD(DATE(COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM)), med.DaysToTake)
    ELSE DATE(COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM))  -- Same day if unknown
  END AS drug_exposure_end_date,
  
  CASE
    WHEN med.EndFuzzySortAs IS NOT NULL THEN med.EndFuzzySortAs
    WHEN med.DaysSupply IS NOT NULL AND med.DaysSupply > 0 
      THEN CAST(DATE_ADD(DATE(COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM)), med.DaysSupply) AS TIMESTAMP)
    WHEN med.DaysToTake IS NOT NULL AND med.DaysToTake > 0 
      THEN CAST(DATE_ADD(DATE(COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM)), med.DaysToTake) AS TIMESTAMP)
    ELSE COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM)
  END AS drug_exposure_end_datetime,
  
  -- Verbatim end date (original source end date)
  DATE(med.EndFuzzySortAs) AS verbatim_end_date,
  
  -- Drug type concept: based on medication context
  CASE
    WHEN med.Samples = 'Y' THEN 32839  -- EHR dispensing
    WHEN med.AdministeredByID IS NOT NULL THEN 32818  -- EHR administration
    ELSE 32838  -- EHR prescription (default)
  END AS drug_type_concept_id,
  
  -- Stop reason
  NULL AS stop_reason,
  
  -- Refills
  med.Refill AS refills,
  
  -- Quantity
  CAST(COALESCE(med.QuantityToDispense, med.SampleQuantity, med.AdministrationDoseQty) AS DOUBLE) AS quantity,
  
  -- Days supply
  COALESCE(med.DaysSupply, med.DaysToTake) AS days_supply,
  
  -- Sig (instructions)
  COALESCE(med.FreeTextSIG, med.Instructions) AS sig,
  
  -- Route concept mapping via domain_source_to_concept
  COALESCE(route_concept.omop_concept_id, 0) AS route_concept_id,
  
  -- Lot number
  COALESCE(med.AdministrationLot, med.SampleLot) AS lot_number,
  
  -- Source values
  COALESCE(med.NDC, med_de.NDC, med_de.DrugName, med_de.DisplayName) AS drug_source_value,
  COALESCE(ndc_concept.concept_id, 0) AS drug_source_concept_id,
  NULL AS route_source_value,  -- Would need to join to route DE table
  COALESCE(med.Dose, med_de.UnitOfMeasure) AS dose_unit_source_value,
  
  -- FK source values (resolved to IDs in gold layer)
  CONCAT('{source}', ' | ', CAST(item_med.PatientID AS STRING)) AS person_source_value,
  CASE 
    WHEN COALESCE(med.PrescribedByID, med.AdministeredByID, med.SupervisedByID, med.WhoDidItID) IS NOT NULL 
    THEN CONCAT('{source}', ' | ', CAST(COALESCE(med.PrescribedByID, med.AdministeredByID, med.SupervisedByID, med.WhoDidItID) AS STRING))
    ELSE NULL 
  END AS provider_source_value,
  NULL AS visit_occurrence_source_value,  -- Would need to trace through order_activity_header
  NULL AS visit_detail_source_value,
  
  -- Unique identifier for this drug exposure
  CONCAT('{source}', ' | ', CAST(med.ID AS STRING)) AS drug_exposure_source_value,
  '{source}' AS source_system

FROM _exponent._bronze_allscripts_tw_works.dbo_medication med

-- Join to medication definition for drug codes
LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works.dbo_medication_de med_de
  ON med.MedDictDE = med_de.ID

-- Join to item_medication for PatientID
INNER JOIN _exponent._bronze_allscripts_tw_works.dbo_item_medication item_med
  ON med.ItemID = item_med.ID
  AND item_med.PatientID IS NOT NULL
  AND item_med.IsErrorFLAG = 'N'

-- Join to OMOP concept table for RxNorm standard concept
LEFT OUTER JOIN _exponent.omop.concept rxnorm_concept
  ON rxnorm_concept.concept_code = COALESCE(med_de.RxNormCodeNormalized, med_de.RxNormCode)
  AND rxnorm_concept.vocabulary_id = 'RxNorm'
  AND rxnorm_concept.standard_concept = 'S'

-- Fallback: NDC to RxNorm mapping
LEFT OUTER JOIN _exponent.omop.concept ndc_concept
  ON ndc_concept.concept_code = COALESCE(med.NDC, med_de.NDC)
  AND ndc_concept.vocabulary_id = 'NDC'

LEFT OUTER JOIN _exponent.omop.concept_relationship ndc_to_rxnorm
  ON ndc_to_rxnorm.concept_id_1 = ndc_concept.concept_id
  AND ndc_to_rxnorm.relationship_id = 'Maps to'

LEFT OUTER JOIN _exponent.omop.concept rxnorm_from_ndc
  ON rxnorm_from_ndc.concept_id = ndc_to_rxnorm.concept_id_2
  AND rxnorm_from_ndc.vocabulary_id = 'RxNorm'
  AND rxnorm_from_ndc.standard_concept = 'S'

-- Route concept mapping via domain_source_to_concept
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept route_concept
  ON route_concept.source_id = CAST(COALESCE(med.RoutOfAdministrationDE, med.AdministrationRouteDE) AS INT)
  AND route_concept.domain_id = 'Route'
  AND route_concept.source_system = '{source}'

-- Only include patients that exist in source_to_person mapping
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT('{source}', ' | ', CAST(item_med.PatientID AS STRING))
  AND stp.active_flag = TRUE

WHERE COALESCE(med.StartFuzzySortAs, med.PerformedFuzzySortAs, med.RecordedDTTM) IS NOT NULL
''')

display(silver_drug_exposure_df)
silver_drug_exposure_df.createOrReplaceTempView("silver_drug_exposure")

In [ ]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.drug_exposure AS t
USING silver_drug_exposure AS s
ON t.drug_exposure_source_value = s.drug_exposure_source_value

WHEN MATCHED AND (
     NOT (t.drug_concept_id <=> s.drug_concept_id)
  OR NOT (t.drug_exposure_start_date <=> s.drug_exposure_start_date)
  OR NOT (t.drug_exposure_start_datetime <=> s.drug_exposure_start_datetime)
  OR NOT (t.drug_exposure_end_date <=> s.drug_exposure_end_date)
  OR NOT (t.drug_exposure_end_datetime <=> s.drug_exposure_end_datetime)
  OR NOT (t.verbatim_end_date <=> s.verbatim_end_date)
  OR NOT (t.drug_type_concept_id <=> s.drug_type_concept_id)
  OR NOT (t.stop_reason <=> s.stop_reason)
  OR NOT (t.refills <=> s.refills)
  OR NOT (t.quantity <=> s.quantity)
  OR NOT (t.days_supply <=> s.days_supply)
  OR NOT (t.sig <=> s.sig)
  OR NOT (t.route_concept_id <=> s.route_concept_id)
  OR NOT (t.lot_number <=> s.lot_number)
  OR NOT (t.drug_source_value <=> s.drug_source_value)
  OR NOT (t.drug_source_concept_id <=> s.drug_source_concept_id)
  OR NOT (t.route_source_value <=> s.route_source_value)
  OR NOT (t.dose_unit_source_value <=> s.dose_unit_source_value)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.drug_concept_id             = s.drug_concept_id,
  t.drug_exposure_start_date    = s.drug_exposure_start_date,
  t.drug_exposure_start_datetime = s.drug_exposure_start_datetime,
  t.drug_exposure_end_date      = s.drug_exposure_end_date,
  t.drug_exposure_end_datetime  = s.drug_exposure_end_datetime,
  t.verbatim_end_date           = s.verbatim_end_date,
  t.drug_type_concept_id        = s.drug_type_concept_id,
  t.stop_reason                 = s.stop_reason,
  t.refills                     = s.refills,
  t.quantity                    = s.quantity,
  t.days_supply                 = s.days_supply,
  t.sig                         = s.sig,
  t.route_concept_id            = s.route_concept_id,
  t.lot_number                  = s.lot_number,
  t.drug_source_value           = s.drug_source_value,
  t.drug_source_concept_id      = s.drug_source_concept_id,
  t.route_source_value          = s.route_source_value,
  t.dose_unit_source_value      = s.dose_unit_source_value,
  t.person_source_value         = s.person_source_value,
  t.provider_source_value       = s.provider_source_value,
  t.visit_occurrence_source_value = s.visit_occurrence_source_value,
  t.visit_detail_source_value   = s.visit_detail_source_value,
  t.source_system               = s.source_system,
  t.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value,
  person_source_value,
  provider_source_value,
  visit_occurrence_source_value,
  visit_detail_source_value,
  drug_exposure_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.drug_concept_id,
  s.drug_exposure_start_date,
  s.drug_exposure_start_datetime,
  s.drug_exposure_end_date,
  s.drug_exposure_end_datetime,
  s.verbatim_end_date,
  s.drug_type_concept_id,
  s.stop_reason,
  s.refills,
  s.quantity,
  s.days_supply,
  s.sig,
  s.route_concept_id,
  s.lot_number,
  s.drug_source_value,
  s.drug_source_concept_id,
  s.route_source_value,
  s.dose_unit_source_value,
  s.person_source_value,
  s.provider_source_value,
  s.visit_occurrence_source_value,
  s.visit_detail_source_value,
  s.drug_exposure_source_value,
  s.source_system,
  current_timestamp()
);

In [ ]:
%sql
-- Verify silver layer
SELECT * FROM _exponent.omop_silver.drug_exposure LIMIT 10

In [ ]:
%sql
-- Insert new mappings to source_to_drug_exposure
INSERT INTO _exponent.omop_mapping.source_to_drug_exposure (
    source_system,
    drug_exposure_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.drug_exposure_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, drug_exposure_source_value, last_mod_tsp
    FROM _exponent.omop_silver.drug_exposure
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_drug_exposure x
  ON s.drug_exposure_source_value = x.drug_exposure_source_value;

In [ ]:
%sql
-- Verify mapping table
SELECT * FROM _exponent.omop_mapping.source_to_drug_exposure LIMIT 10

In [ ]:
%sql
-- Merge to Gold layer
-- Note: provider_id and visit_occurrence_id are NULL (mapping tables not created)
MERGE INTO _exponent.omop.drug_exposure AS gold
USING (
  SELECT
    sde.drug_exposure_id,
    stp.person_id,
    s.drug_concept_id,
    s.drug_exposure_start_date,
    s.drug_exposure_start_datetime,
    s.drug_exposure_end_date,
    s.drug_exposure_end_datetime,
    s.verbatim_end_date,
    s.drug_type_concept_id,
    s.stop_reason,
    s.refills,
    s.quantity,
    s.days_supply,
    s.sig,
    s.route_concept_id,
    s.lot_number,
    NULL AS provider_id,  -- source_to_provider not yet created
    NULL AS visit_occurrence_id,  -- source_to_visit_occurrence not yet created
    NULL AS visit_detail_id,  -- source_to_visit_detail not yet created
    s.drug_source_value,
    s.drug_source_concept_id,
    s.route_source_value,
    s.dose_unit_source_value
  FROM _exponent.omop_silver.drug_exposure s
  JOIN _exponent.omop_mapping.source_to_drug_exposure sde
    ON sde.drug_exposure_source_value = s.drug_exposure_source_value
   AND sde.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
) AS src
ON gold.drug_exposure_id = src.drug_exposure_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                    = src.person_id,
  gold.drug_concept_id              = src.drug_concept_id,
  gold.drug_exposure_start_date     = src.drug_exposure_start_date,
  gold.drug_exposure_start_datetime = src.drug_exposure_start_datetime,
  gold.drug_exposure_end_date       = src.drug_exposure_end_date,
  gold.drug_exposure_end_datetime   = src.drug_exposure_end_datetime,
  gold.verbatim_end_date            = src.verbatim_end_date,
  gold.drug_type_concept_id         = src.drug_type_concept_id,
  gold.stop_reason                  = src.stop_reason,
  gold.refills                      = src.refills,
  gold.quantity                     = src.quantity,
  gold.days_supply                  = src.days_supply,
  gold.sig                          = src.sig,
  gold.route_concept_id             = src.route_concept_id,
  gold.lot_number                   = src.lot_number,
  gold.provider_id                  = src.provider_id,
  gold.visit_occurrence_id          = src.visit_occurrence_id,
  gold.visit_detail_id              = src.visit_detail_id,
  gold.drug_source_value            = src.drug_source_value,
  gold.drug_source_concept_id       = src.drug_source_concept_id,
  gold.route_source_value           = src.route_source_value,
  gold.dose_unit_source_value       = src.dose_unit_source_value

WHEN NOT MATCHED THEN INSERT (
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  drug_exposure_start_datetime,
  drug_exposure_end_date,
  drug_exposure_end_datetime,
  verbatim_end_date,
  drug_type_concept_id,
  stop_reason,
  refills,
  quantity,
  days_supply,
  sig,
  route_concept_id,
  lot_number,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  drug_source_value,
  drug_source_concept_id,
  route_source_value,
  dose_unit_source_value
)
VALUES (
  src.drug_exposure_id,
  src.person_id,
  src.drug_concept_id,
  src.drug_exposure_start_date,
  src.drug_exposure_start_datetime,
  src.drug_exposure_end_date,
  src.drug_exposure_end_datetime,
  src.verbatim_end_date,
  src.drug_type_concept_id,
  src.stop_reason,
  src.refills,
  src.quantity,
  src.days_supply,
  src.sig,
  src.route_concept_id,
  src.lot_number,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.drug_source_value,
  src.drug_source_concept_id,
  src.route_source_value,
  src.dose_unit_source_value
);

In [ ]:
%sql
-- Verify gold layer
SELECT * FROM _exponent.omop.drug_exposure LIMIT 10

In [ ]:
%sql
-- Validation: Count records at each layer
SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.drug_exposure
UNION ALL
SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_drug_exposure
UNION ALL
SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.drug_exposure

In [ ]:
%sql
-- Data Quality: Check unmapped drugs (concept_id = 0)
SELECT 
  COUNT(*) AS total_records,
  SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_count,
  ROUND(100.0 * SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS unmapped_pct
FROM _exponent.omop.drug_exposure